## Imports

In [ ]:
%load_ext autoreload
%autoreload 2
# Internal import
import deep_learning_land_use_classification.config as config
import deep_learning_land_use_classification.evaluation as evaluation
from deep_learning_land_use_classification.dataset import get_single_label_data
import deep_learning_land_use_classification.wanddb_helpers as wh

import torch
from transformers import AutoImageProcessor, AutoModel

from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
import wandb
import torch.nn as nn

c:\Users\tomer\anaconda3\envs\Deep-learning-land-use-classification\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
train_df, test_df, val_df, class_names, num_classes = get_single_label_data()

In [ ]:
# Wandb initialization for experiment tracking
run = wh.init_run(
    task="single",
    architecture="dinov3-vitl16",
    num_classes=num_classes,
    loss="CrossEntropyLoss",
    epochs=5,
    batch_size=32,
    learning_rate=1e-4,
    optimizer="AdamW",
    pretrained=True,
    pretraining_dataset="sat493m",
    pretraining_source="huggingface",
    weights="facebook/dinov3-vitl16-pretrain-sat493m",
    model_name=None,
    augmentation=False,
    early_stopping=True,
    patience=2,
    min_delta=0.001
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\tomer\_netrc.
wandb: Currently logged in as: tomer-peled (sstaheli52-wageningen-university-and-research) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
pretrained_model_name = "facebook/dinov3-vitl16-pretrain-sat493m"
processor = AutoImageProcessor.from_pretrained(pretrained_model_name)

In [5]:
print(train_df.head())

                                                   path       label
1592  C:\Users\tomer\OneDrive\Home\Projects\Deep-lea...  parkinglot
829   C:\Users\tomer\OneDrive\Home\Projects\Deep-lea...     freeway
414   C:\Users\tomer\OneDrive\Home\Projects\Deep-lea...   buildings
704   C:\Users\tomer\OneDrive\Home\Projects\Deep-lea...      forest
755   C:\Users\tomer\OneDrive\Home\Projects\Deep-lea...      forest


In [19]:
class SingleLabelDataset(Dataset):
    def __init__(self, dataframe, class_names, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.class_names = class_names
        self.transform = transform
        self.label_to_idx = {label: i for i, label in enumerate(class_names)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["path"]).convert("RGB")

        if self.transform is None:
            raise ValueError("SingleLabelDataset requires a transform/processor.")
        inputs = self.transform(images=image, return_tensors="pt")
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}

        label = torch.tensor(self.label_to_idx[row["label"]], dtype=torch.long)

        return inputs, label

In [ ]:
# Create datasets and dataloaders
train_dataset = SingleLabelDataset(train_df, class_names, processor)
val_dataset  = SingleLabelDataset(val_df, class_names, processor)

train_loader = DataLoader(train_dataset, batch_size=run.config.batch_size, shuffle=True)
val_loader  = DataLoader(val_dataset, batch_size=run.config.batch_size, shuffle=False)


In [8]:
# this is a simple wrapper around the DINO backbone to add a classification head on top
class DinoClassifier(torch.nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        self.classifier = torch.nn.Linear(
            backbone.config.hidden_size,
            num_classes
        )

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values) # get the output of the backbone
        features = outputs.pooler_output
        return self.classifier(features)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("Using device:", device)

backbone = AutoModel.from_pretrained(pretrained_model_name)

model = DinoClassifier(backbone, num_classes=num_classes).to(device)
model = model.to(device)


Using device: cuda


Loading weights: 100%|██████████| 415/415 [00:00<00:00, 3299.33it/s]


In [17]:
# Compute class weights for single-label classification
label_to_idx = {name: i for i, name in enumerate(class_names)}
label_idx = train_df["label"].map(label_to_idx).to_numpy()

class_counts = np.bincount(label_idx, minlength=num_classes).astype(np.float32)
class_weights = class_counts.sum() / (num_classes * (class_counts + 1e-5))
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

for p in model.backbone.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(
    model.classifier.parameters(),  # only head
    lr=run.config.learning_rate,
    weight_decay=0.01
)

In [ ]:
wh.log_model_summary(run, model)

### Train Model

In [14]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for inputs, labels in loader:
        pixel_values = inputs["pixel_values"].to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(pixel_values)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [15]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for inputs, labels in loader:
            pixel_values = inputs["pixel_values"].to(device)
            labels = labels.to(device)

            outputs = model(pixel_values)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
# Early stopping class
class EarlyStopper:
    def __init__(self, patience=2, min_delta=0.001):
        self.patience = patience      # epochs to wait after last improvement
        self.min_delta = min_delta    # minimum change to count as improvement
        self.counter = 0
        self.best_loss = float('inf')
        self.best_weights = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            # Save a copy of the best weights
            self.best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1

        return self.counter >= self.patience  # True = stop training

    def restore_best_weights(self, model):
        if self.best_weights:
            model.load_state_dict(self.best_weights)

In [ ]:
epochs = run.config.epochs
early_stopper = EarlyStopper(patience=run.config.patience, min_delta=run.config.min_delta)

for epoch in range(epochs):
    train_loss = train(model, train_loader, optimizer, criterion)
    val_loss  = evaluate(model, val_loader, criterion)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"val Loss:  {val_loss:.4f}")
    precision, recall, f1, p_macro, r_macro, f1_macro = evaluation.compute_accuracy_metrics_singlelabel(
        model,
        val_loader,
        device,
        num_classes=num_classes,
    )
        
    wh.log_epoch(run, epoch, train_loss, val_loss,
        precision, recall, f1, p_macro, r_macro, f1_macro, class_names)
    
    # --- Early stopping check ---
    if early_stopper.step(val_loss, model):
        print(f"Early stopping triggered at epoch {epoch+1}. Best val loss: {early_stopper.best_loss:.4f}")
        break

# Restore the weights from the best epoch
early_stopper.restore_best_weights(model)
print("Restored best model weights.")

run.finish()

Epoch 1/5
Train Loss: 2.9094
Test Loss:  2.7396
Epoch 2/5
Train Loss: 2.5918
Test Loss:  2.4563
Epoch 3/5
Train Loss: 2.3211
Test Loss:  2.2096
Epoch 4/5
Train Loss: 2.0892
Test Loss:  2.0046
Epoch 5/5
Train Loss: 1.8974
Test Loss:  1.8348
